In [1]:
!pip install torch torchvision numpy pandas scikit-learn opencv-python


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import cv2
import time

from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 128
BATCH_SIZE = 32


In [4]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)


100%|██████████| 26.4M/26.4M [00:02<00:00, 11.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 193kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.56MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 22.5MB/s]


In [20]:
data = []

for img, label in dataset:
    img_np = (img.numpy().squeeze() * 255).astype(np.uint8)

    if label == 0:
        mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        anomaly_label = 0
    else:
        mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        cv2.rectangle(mask, (40,40), (90,90), 255, -1)
        anomaly_label = 1

    data.append({
        "img": img_np,          # (H, W)
        "mask": mask,           # (H, W)
        "label": anomaly_label
    })

df = pd.DataFrame(data)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"])


In [21]:
class FashionAnomalyDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = self.df.loc[idx, "img"]      # shape: (H, W)
        mask = self.df.loc[idx, "mask"]
        label = self.df.loc[idx, "label"]

        img_t = torch.tensor(img).unsqueeze(0).float() / 255.0  # (1, H, W)
        mask_t = torch.tensor(mask).unsqueeze(0).float() / 255.0
        label_t = torch.tensor([label]).float()

        return img_t, mask_t, label_t


In [13]:
class ModelA_Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),  # ← FIX
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [17]:
class ModelB_Localizer(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),  # ← FIX
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 1, 2, stride=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


In [15]:
class SequentialController:
    def __init__(self, model_a, model_b):
        self.model_a = model_a.eval().to(DEVICE)
        self.model_b = model_b.eval().to(DEVICE)
        self.logs = []

    def run(self, dataset):
        for i in range(len(dataset)):
            img, _, label = dataset[i]
            img = img.unsqueeze(0).to(DEVICE)

            entry = {"Index": i, "Actual": int(label.item())}

            # Stage 1
            t0 = time.perf_counter()
            with torch.no_grad():
                anomaly = self.model_a(img).item() > 0.5
            entry["Stage1_ms"] = round((time.perf_counter() - t0) * 1000, 2)

            # Stage 2
            entry["Stage2_ms"] = 0.0
            if anomaly:
                t1 = time.perf_counter()
                with torch.no_grad():
                    _ = self.model_b(img)
                entry["Stage2_ms"] = round((time.perf_counter() - t1) * 1000, 2)
                entry["Decision"] = "ALERT"
            else:
                entry["Decision"] = "PASS"

            entry["Total_ms"] = entry["Stage1_ms"] + entry["Stage2_ms"]
            self.logs.append(entry)

    def report(self):
        return pd.DataFrame(self.logs)


In [22]:
test_dataset = FashionAnomalyDataset(test_df)

model_a = ModelA_Classifier().to(DEVICE)
model_b = ModelB_Localizer().to(DEVICE)

controller = SequentialController(model_a, model_b)
controller.run(test_dataset)

report = controller.report()
report.head()


,Index,Actual,Stage1_ms,Stage2_ms,Decision,Total_ms
0,0,1,995.88,37.17,ALERT,1033.05
1,1,1,0.70,0.31,ALERT,1.01
2,2,0,0.53,0.23,ALERT,0.76
3,3,1,0.40,0.21,ALERT,0.61
4,4,1,0.40,0.22,ALERT,0.62


In [23]:
print("="*60)
print("SEQUENTIAL ANOMALY INSPECTION REPORT")
print("="*60)
print(report.to_string(index=False))
print("="*60)
print("Average Latency:", report["Total_ms"].mean(), "ms")


Streaming output truncated to the last 5000 lines.
  7002       1       0.42       0.23    ALERT      0.65
  7003       1       0.39       0.23    ALERT      0.62
  7004       1       0.41       0.25    ALERT      0.66
  7005       0       0.40       0.26    ALERT      0.66
  7006       1       0.40       0.28    ALERT      0.68
  7007       1       0.40       0.26    ALERT      0.66
  7008       0       0.40       0.25    ALERT      0.65
  7009       1       0.41       0.27    ALERT      0.68
  7010       1       0.36       0.22    ALERT      0.58
  7011       1       0.42       0.30    ALERT      0.72
  7012       0       0.45       0.25    ALERT      0.70
  7013       1       0.40       0.23    ALERT      0.63
  7014       1       0.40       0.23    ALERT      0.63
  7015       1       0.42       0.23    ALERT      0.65
  7016       1       0.46       0.25    ALERT      0.71
  7017       1       0.39       0.26    ALERT      0.65
  7018       1       0.41       0.25    ALERT      0.